# Classical CLIP Embedding Extraction

This notebook extracts 768-dim classical CLIP features for video frames.

### Workflow Stage:
1. **Classical Extraction**: CLIP-ViT-L/14 extracts 768-dim features.
2. **Output**: Raw embeddings saved for later Quantum-Enhanced training.

In [1]:
import os
import glob
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel

# Configuration
DATA_DIR = "../data/Videos"
OUTPUT_BASE_DIR = "../embeddings/QCLIP"
BATCH_SIZE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLIP_MODEL_NAME = "openai/clip-vit-large-patch14"

print(f"Using device: {DEVICE}")
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

C:\Users\tejes\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## 1. Classical Feature Extraction (CLIP-ViT-L/14)

In [2]:
print("Loading CLIP model...")
model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
model.eval()

def extract_clip_features(image_paths):
    features = []
    for i in range(0, len(image_paths), BATCH_SIZE):
        batch_paths = image_paths[i:i+BATCH_SIZE]
        images = [Image.open(p).convert("RGB") for p in batch_paths]
        inputs = processor(images=images, return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            batch_features = model.get_image_features(**inputs)
            if not isinstance(batch_features, torch.Tensor):
                if hasattr(batch_features, "image_embeds"):
                    batch_features = batch_features.image_embeds
                elif hasattr(batch_features, "pooler_output"):
                    batch_features = batch_features.pooler_output
        features.append(batch_features.cpu().numpy())
    return np.concatenate(features, axis=0)

Loading CLIP model...


Loading weights: 100%|██████████| 590/590 [00:00<00:00, 5207.25it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 2. Main Processing Pipeline

In [3]:
video_ids = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

pbar = tqdm(video_ids, desc="Processing Videos")
for vid in pbar:
    vid_path = os.path.join(DATA_DIR, vid)
    segments = sorted([s for s in os.listdir(vid_path) if os.path.isdir(os.path.join(vid_path, s))])
    
    for seg in segments:
        seg_path = os.path.join(vid_path, seg)
        out_seg_path = os.path.join(OUTPUT_BASE_DIR, vid, seg)
        
        # Check if already processed
        if os.path.exists(os.path.join(out_seg_path, "embeddings.npy")):
            continue
            
        frame_paths = sorted(glob.glob(os.path.join(seg_path, "*.jpg")))
        if not frame_paths: continue
            
        # 1. Classical CLIP Extraction (768-dim)
        clip_feats = extract_clip_features(frame_paths)
        
        # Save Raw Classical Results
        os.makedirs(out_seg_path, exist_ok=True)
        np.save(os.path.join(out_seg_path, "embeddings.npy"), clip_feats)
        
    pbar.set_postfix(video=vid)

print("All Classical CLIP embeddings extracted and saved successfully.")

Processing Videos: 100%|██████████| 411/411 [1:33:15<00:00, 13.61s/it, video=ztgJ33SMons]

All Classical CLIP embeddings extracted and saved successfully.
